# Task 4: Text Vectorization Using TF-IDF and Similarity Analysis

**Project:** TED Talks Recommendation System  
**Notebook file:** `04_tfidf_vectorization.ipynb`

## What are we doing?
In the previous notebook, we cleaned the TED Talks data and created a `combined_text` column using the talk title, description, tags, and transcript. In this notebook, we continue from that work.

We will:
1. Load the cleaned dataset.
2. Convert each talk's text into numbers using TF-IDF.
3. Compare talks using cosine similarity.
4. Show the top 5 similar talks for at least 3 different talks.
5. Review results and limitations.

**Important:** Recommendations are generated from the dataset you run this notebook with. A high similarity score means the text is similar; it does not guarantee that a person will like the talk.

## 1. Concepts in simple words

- **Machine learning lifecycle:** prepare data → create numerical features → build/use a method → evaluate results → deploy it.
- **NLP:** techniques that help computers work with human language.
- **Document:** one TED Talk's text.
- **Corpus:** the collection of all TED Talks in our dataset.
- **Token:** a word or text unit, such as `climate`.
- **Vocabulary:** the set of terms the vectorizer learns from all documents.
- **Feature extraction:** converting text into numbers that a computer can compare.
- **TF (Term Frequency):** how often a term appears in one talk's text.
- **IDF (Inverse Document Frequency):** gives less weight to terms that appear in many documents and more weight to terms that are more distinctive.
- **TF-IDF:** combines TF and IDF, giving a score to each term in each talk.
- **Vector:** a row of numerical TF-IDF values representing one talk.
- **Cosine similarity:** compares the direction of two vectors. A higher score usually means more overlap in their weighted words.
- **Ranking:** sorting candidate talks from the highest similarity score to the lowest.

### Where this notebook fits in the project

`Cleaned text → TF-IDF vectors → cosine similarity → ranked similar talks`

We use TF-IDF and cosine similarity as a content-based recommendation method. This is not a trained classifier and it does not learn whether a user personally likes a talk.

## 2. Import libraries

We use `pandas` to work with the dataset, `TfidfVectorizer` to create numerical text features, and `cosine_similarity` to compare talks.

In [ ]:
import os
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_colwidth", 120)

## 3. Load the cleaned dataset

The required location is `data/processed/ted_talks_cleaned.csv`.

If you are using Google Colab, upload the cleaned CSV to the session and make sure it is saved at that path. If the file is not found, this cell lets you upload it. The CSV should contain at least `title` and `combined_text`, created in the previous project notebook.

In [ ]:
# Required path from the task instructions
data_path = "data/processed/ted_talks_cleaned.csv"

# If the file is not present in the current Colab session, ask you to upload it.
if not os.path.exists(data_path):
    print(f"File not found at: {data_path}")
    print("Upload ted_talks_cleaned.csv. It must contain 'title' and 'combined_text' columns.")

    try:
        from google.colab import files
        uploaded = files.upload()

        # Find the uploaded CSV file
        csv_files = [name for name in uploaded.keys() if name.lower().endswith(".csv")]
        if not csv_files:
            raise FileNotFoundError("No CSV file was uploaded.")

        uploaded_path = csv_files[0]
        df = pd.read_csv(uploaded_path)

        # Create the requested folder and save a copy at the required path
        os.makedirs(os.path.dirname(data_path), exist_ok=True)
        df.to_csv(data_path, index=False)
        print(f"Saved uploaded file to: {data_path}")
    except ImportError:
        raise FileNotFoundError(
            f"Please place the cleaned CSV at '{data_path}' before running this notebook."
        )
else:
    df = pd.read_csv(data_path)

# Load the file if it was just saved by the upload step
df = pd.read_csv(data_path)

print("Dataset loaded successfully.")
print("Number of rows and columns:", df.shape)

## 4. Review the dataset

First, inspect column names and confirm the two required fields are present. We also check for empty titles/text and remove records that cannot be used for text similarity.

In [ ]:
# Display all available columns
print("Available columns:")
print(df.columns.tolist())

# Confirm the required columns exist
required_columns = ["title", "combined_text"]
missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}. "
        "Return to the previous notebook and create the combined_text column."
    )

# Keep only rows with a usable title and usable combined text
df["title"] = df["title"].fillna("").astype(str).str.strip()
df["combined_text"] = df["combined_text"].fillna("").astype(str).str.strip()

before_rows = len(df)
df = df[(df["title"] != "") & (df["combined_text"] != "")].copy()
df = df.reset_index(drop=True)

print("Rows before removing empty title/text:", before_rows)
print("Usable talk records:", len(df))
display(df[["title", "combined_text"]].head(3))

## 5. Create the TF-IDF matrix

We use the same type of settings introduced in the previous project notebook:
- `max_features=10000`: keep up to 10,000 terms.
- `stop_words="english"`: ignore common English words such as “the” and “is”.
- `ngram_range=(1, 2)`: consider single words and pairs of adjacent words.
- `min_df=2`: ignore terms appearing in fewer than two talks.
- `max_df=0.95`: ignore terms appearing in more than 95% of talks.
- `sublinear_tf=True`: reduce the effect of a word appearing many times in one document.

The exact vocabulary and matrix size depend on the dataset.

In [ ]:
tfidf = TfidfVectorizer(
    max_features=10000,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

# Each row represents one TED Talk.
# Each column represents one learned word or two-word phrase.
# Each value is that term's TF-IDF weight for that talk.
X_tfidf = tfidf.fit_transform(df["combined_text"])

print("Number of talks (rows):", X_tfidf.shape[0])
print("Number of text features (columns):", X_tfidf.shape[1])
print("TF-IDF matrix dimensions:", X_tfidf.shape)
print("Number of stored non-zero values:", X_tfidf.nnz)

### How to interpret the matrix

If the shape is `(1000, 10000)`, that means 1,000 talks and 10,000 text features. This is only an example; your output will depend on your dataset.

Most talk/term combinations have a value of zero, so scikit-learn stores this as a **sparse matrix** to save memory. A larger TF-IDF value means a term has a stronger weight in that talk; it is not a probability.

In [ ]:
# Show a small sample of the vocabulary learned by TF-IDF
terms = tfidf.get_feature_names_out()

print("First 30 vocabulary terms:")
print(terms[:30].tolist())

# Show the 15 terms with the highest TF-IDF weights for the first talk
first_vector = X_tfidf[0].toarray().ravel()
top_term_indices = first_vector.argsort()[::-1][:15]

top_terms_first_talk = pd.DataFrame({
    "term": terms[top_term_indices],
    "tfidf_weight": first_vector[top_term_indices]
})

print("\nTalk used for this example:", df.loc[0, "title"])
display(top_terms_first_talk[top_terms_first_talk["tfidf_weight"] > 0])

## 6. Compare talks using cosine similarity

Cosine similarity compares two TF-IDF vectors. A score closer to 1 means their weighted word patterns are more similar; a score near 0 means they have little overlap in this representation.

We do not recommend a talk to itself. A talk's similarity with itself is normally 1.0, so we exclude the selected talk from the candidate results.

In [ ]:
# Calculate pairwise cosine similarity between all talk vectors.
# Result: a square matrix where entry [i, j] is similarity between talk i and talk j.
similarity_matrix = cosine_similarity(X_tfidf)

print("Similarity matrix dimensions:", similarity_matrix.shape)

# Show similarities between the first talk and the next few talks as a small example
sample_end = min(6, len(df))
sample_similarity = pd.DataFrame(
    similarity_matrix[0:1, 0:sample_end],
    columns=df["title"].iloc[0:sample_end]
)
print("Similarity scores for the first talk against a small sample:")
display(sample_similarity)

## 7. Function to recommend the top 5 similar talks

This function:
1. Finds the selected talk.
2. Reads its similarity scores against all other talks.
3. Removes the selected talk itself.
4. Sorts the remaining talks from highest score to lowest.
5. Returns up to five recommendations.

If the dataset contains fewer than six usable talks, fewer than five recommendations may be available.

In [ ]:
def recommend_similar_talks(talk_index, top_n=5):
    # Get the similarity scores for the selected talk against every talk
    scores = similarity_matrix[talk_index].copy()

    # A talk must not recommend itself
    scores[talk_index] = -1

    # Sort talk indices by similarity score, highest first
    ranked_indices = np.argsort(scores)[::-1]

    # Keep only the requested number of recommendations
    ranked_indices = ranked_indices[:min(top_n, len(df) - 1)]

    recommendations = pd.DataFrame({
        "selected_talk": df.loc[talk_index, "title"],
        "recommended_talk": df.loc[ranked_indices, "title"].values,
        "similarity_score": scores[ranked_indices]
    })

    return recommendations

## 8. Generate recommendations for three different TED Talks

To make this work with your actual dataset, the code selects three distinct talks from the loaded data. It uses the first, middle, and last available records. You can replace these indices with titles you want to investigate.

Review the selected titles and results carefully. The code does not assume the recommendations are relevant; that is something you assess manually.

In [ ]:
if len(df) < 2:
    raise ValueError("At least two usable talks are needed for similarity recommendations.")

# Pick up to three different talk indices spread across the dataset
candidate_indices = [0, len(df) // 2, len(df) - 1]
selected_indices = []
for idx in candidate_indices:
    if idx not in selected_indices:
        selected_indices.append(idx)

print("Selected talks for analysis:")
for idx in selected_indices:
    print(f"- Row {idx}: {df.loc[idx, 'title']}")

recommendation_tables = []

for idx in selected_indices:
    print("\n" + "=" * 90)
    print("SELECTED TALK:", df.loc[idx, "title"])
    results = recommend_similar_talks(idx, top_n=5)
    display(results)
    recommendation_tables.append(results)

# Combine all recommendation results into one table for the deliverable
all_recommendations = pd.concat(recommendation_tables, ignore_index=True)

## 9. Save the recommendation results

This saves the recommendation tables as a CSV file so you can include them in your task submission or report.

In [ ]:
os.makedirs("data/processed", exist_ok=True)
recommendation_output = "data/processed/ted_talk_similarity_recommendations.csv"
all_recommendations.to_csv(recommendation_output, index=False)

print("Saved recommendation results to:", recommendation_output)
display(all_recommendations)

## 10. Manually review recommendation quality

Use the table above and fill in your own observations. Do not automatically mark a recommendation as good just because its score is high.

For example, check whether:
- the selected talk and recommended talk discuss related subjects;
- the same keywords are used in different contexts;
- talks that seem related use different words and therefore receive lower scores.

The following template is for your own review. Replace the example text with what you actually observe in your results.

In [ ]:
# Fill this table after reading your actual recommendations.
manual_review = pd.DataFrame([
    {
        "selected_talk": df.loc[selected_indices[0], "title"],
        "recommended_talk": all_recommendations[
            all_recommendations["selected_talk"] == df.loc[selected_indices[0], "title"]
        ]["recommended_talk"].iloc[0],
        "seems_relevant": "Review manually",
        "reason": "Write why the subjects appear related or unrelated."
    }
])

display(manual_review)

## 11. Limitations and observations

Write at least three observations based on the output you actually saw. These are possible limitations to check, not claims about your particular results:

1. **Word overlap is not the same as meaning.** Two talks can share keywords but discuss them in different ways.
2. **Different vocabulary can hide a relationship.** Two talks may discuss the same idea using different words, so their score can be low.
3. **No user preference is included.** The method ranks text similarity; it does not know a user's interests, history, or satisfaction.
4. **Transcript length and text quality can affect results.** Longer or noisy text may change which terms receive weight.
5. **A high score is not a guarantee.** Similarity is a mathematical comparison, not proof that the recommendation is useful or enjoyable.

### Your actual observations

After running the notebook, add three or more specific observations here. Mention talk titles or scores from your output so the notes are based on your dataset.

## 12. Short explanation for submission

You can use this as a starting point and rewrite it in your own words:

**TF-IDF:** TF-IDF converts the text of each TED Talk into a numerical vector. It gives more weight to words that are useful for distinguishing talks and less weight to words that appear in many talks. Each row of the matrix represents a talk, and each column represents a vocabulary term or phrase.

**Cosine similarity:** Cosine similarity compares two TF-IDF vectors and returns a score. We sort the scores from highest to lowest to find the most textually similar talks. We exclude the selected talk itself because it would otherwise appear as its own highest-scoring recommendation.

**Project use:** These ranked results are the content-based recommendation stage of our TED Talks Recommendation System. They provide candidate recommendations, which should still be reviewed for relevance.

## 13. Completion checklist

- [ ] Loaded `data/processed/ted_talks_cleaned.csv`
- [ ] Confirmed `title` and `combined_text`
- [ ] Printed TF-IDF matrix dimensions
- [ ] Inspected a sample of vocabulary terms
- [ ] Calculated cosine similarity
- [ ] Generated recommendations for at least three talks
- [ ] Generated up to five recommendations per talk
- [ ] Saved recommendation results
- [ ] Manually reviewed results and wrote at least three observations